## Въведение 

Този урок ще обхване: 
- Какво е извикване на функция и кога се използва 
- Как да създадете извикване на функция с Azure OpenAI 
- Как да интегрирате извикване на функция в приложение 

## Цели на обучението 

След завършване на този урок ще знаете как да и ще разбирате: 

- Целта на използването на извикване на функция 
- Настройване на извикване на функция с Azure Open AI Service 
- Проектиране на ефективни извиквания на функции за вашия случай на употреба в приложения 


## Разбиране на извикванията на функции 

За този урок искаме да създадем функция за нашия образователен стартъп, която да позволява на потребителите да използват чатбот за намиране на технически курсове. Ще препоръчваме курсове, които отговарят на нивото им на умения, текущата им роля и технологиите, които ги интересуват. 

За да изпълним това, ще използваме комбинация от: 
 - `Azure Open AI` за създаване на чат изживяване за потребителя
 - `Microsoft Learn Catalog API` за помощ на потребителите да намерят курсове според заявката на потребителя 
 - `Function Calling` за взимане на заявката на потребителя и изпращането ѝ към функция за извършване на API заявката. 

За да започнем, нека разгледаме защо бихме искали да използваме извикване на функция от самото начало: 


### Защо извикване на функции 

Ако сте завършили който и да е друг урок в този курс, вероятно разбирате силата на използване на големи езикови модели (LLM). Надяваме се също да можете да видите някои от техните ограничения. 

Извикването на функции е функция в Azure Open AI Service, която преодолява следните ограничения: 
1) Консистентен формат на отговора 
2) Възможност за използване на данни от други източници на приложение в контекст на чат 

Преди извикването на функции отговорите от LLM бяха неструктурирани и несъгласувани. Разработчиците трябваше да пишат сложен код за проверка, за да се уверят, че могат да обработят всяка вариация на отговора. 

Потребителите не можеха да получат отговори на въпроси като "Какво е текущото време в Стокхолм?". Това е защото моделите бяха ограничени до времето, когато са били обучени на данните. 

Нека разгледаме примера по-долу, който илюстрира този проблем: 

Да кажем, че искаме да създадем база данни със студентски данни, за да можем да им предложим подходящия курс. По-долу имаме две описания на студенти, които са много подобни по данните, които съдържат.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Искаме да изпратим това към голям езиков модел (LLM), за да анализира данните. Това по-късно може да се използва в нашето приложение, за да бъде изпратено към API или съхранено в база данни. 

Нека създадем два идентични подсказки, с които ще инструктираме LLM каква информация ни интересува: 


Искаме да изпратим това на LLM, за да анализира частите, които са важни за нашия продукт. Така можем да създадем два идентични подсказки за инструктиране на LLM: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


След създаването на тези два подсказвания, ще ги изпратим към LLM, като използваме `client.responses.create`. Съхраняваме подсказването в променливата `input` и задаваме ролята на `user`. Това е, за да се имитира съобщение от потребител, написано към чатбот. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Сега можем да изпратим и двата запитвания към LLM и да прегледаме отговора, който получаваме. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Въпреки че подсказките са еднакви и описанията са сходни, можем да получим различни формати на свойството `Grades`.

Ако изпълните горната клетка няколко пъти, форматът може да бъде `3.7` или `3.7 GPA`.

Това е така, защото LLM приема неструктурирани данни под формата на написаната подсказка и връща също неструктурирани данни. Нужен ни е структуриран формат, за да знаем какво да очакваме при съхраняване или използване на тези данни.

Чрез използването на функционално извикване можем да се уверим, че получаваме обратно структурираните данни. При използване на функционално извикване, LLM всъщност не извършва или изпълнява никакви функции. Вместо това създаваме структура, която LLM трябва да следва за своите отговори. След това използваме тези структурираните отговори, за да знаем коя функция да изпълним в нашите приложения.
 


![Диаграма на потока на извикване на функция](../../../../translated_images/bg/Function-Flow.083875364af4f4bb.webp)


След това можем да вземем това, което функцията връща, и да го изпратим обратно към LLM. LLM ще отговори, използвайки естествен език, за да отговори на запитването на потребителя. 


### Приложения за използване на извиквания на функции

**Извикване на външни инструменти**
Чатботовете са отлични в предоставянето на отговори на въпроси от потребители. Като използват извикване на функция, чатботовете могат да използват съобщения от потребителите, за да изпълнят определени задачи. Например, студент може да поиска от чатбота „Изпрати имейл на моя преподавател, че имам нужда от повече помощ по този предмет“. Това може да направи извикване на функция  `send_email(to: string, body: string)`


**Създаване на заявки към API или база данни**
Потребителите могат да намерят информация, използвайки естествен език, която се преобразува в форматирана заявка или API заявка. Пример за това може да бъде учител, който пита "Кои са студентите, които завършиха последното задание", което може да извика функция на име `get_completed(student_name: string, assignment: int, current_status: string)`


**Създаване на структурирани данни**
Потребителите могат да вземат текстов блок или CSV и да използват LLM, за да извлекат важна информация от него. Например, студент може да преобразува Wikipedia статия за мирни споразумения, за да създаде AI флаш карти. Това може да стане чрез използване на функция, наречена `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Създаване на първото ви извикване на функция  

Процесът на създаване на извикване на функция включва 3 основни стъпки:  
1. Извикване на Chat Completions API с списък на вашите функции и съобщение от потребителя  
2. Прочитане на отговора на модела за предприемане на действие, т.е. изпълнение на функция или API извикване  
3. Направете друго извикване към Chat Completions API с отговора от вашата функция, за да използвате тази информация за създаване на отговор към потребителя.  


![Поток на повикване на функция](../../../../translated_images/bg/LLM-Flow.3285ed8caf4796d7.webp)


### Елементи на извикване на функция 

#### Вход от потребителя 

Първата стъпка е да създадете потребителско съобщение. Това може да бъде динамично зададено, като се вземе стойността от текстов вход, или можете да зададете стойност тук. Ако е първият ви път да работите с Chat Completions API, трябва да определим `role` и `content` на съобщението. 

`role` може да бъде `system` (създаване на правила), `assistant` (моделът) или `user` (крайният потребител). За извикване на функция ще зададем това като `user` и примерен въпрос. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Създаване на функции. 

Следва да дефинираме функция и параметрите на тази функция. Ще използваме само една функция тук, наречена `search_courses`, но можете да създавате множество функции.

**Важно** : Функциите са включени в системното съобщение към LLM и ще се отчитат в броя на наличните токени, с които разполагате. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Дефиниции** 

`name` - Името на функцията, която искаме да бъде извикана. 

`description` - Това е описанието на това как работи функцията. Тук е важно да бъдем конкретни и ясни 

`parameters` - Списък на стойности и формати, които искате моделът да използва в отговора си 


`type` - Типът данни, в който ще се съхраняват свойствата 

`properties` - Списък на конкретните стойности, които моделът ще използва за своя отговор 


`name` - името на свойството, което моделът ще използва във форматирания си отговор 

`type` - Типът данни на това свойство 

`description` - Описание на конкретното свойство 


**По избор**

`required` - задължително свойство за завършване на повикването на функцията 


### Извикване на функцията 
След като дефинираме функция, сега трябва да я включим в извикването на Chat Completion API. Правим това, като добавим `functions` към заявката. В този случай `functions=functions`. 

Има и опция да зададем `function_call` на `auto`. Това означава, че ще оставим LLM да реши коя функция трябва да бъде извикана въз основа на съобщението на потребителя, вместо да я присвояваме ние.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Сега нека разгледаме отговора и да видим как е форматиран: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Виждате, че е извиканото името на функцията и от съобщението на потребителя, LLM успя да намери данните, които да отговарят на аргументите на функцията. 


## 3. Интегриране на извиквания на функции в приложение. 


След като сме тествали форматирания отговор от LLM, сега можем да го интегрираме в приложение. 

### Управление на потока 

За да го интегрираме в нашето приложение, нека предприемем следните стъпки: 

Първо, нека направим извикване към Open AI услугите и съхраним съобщението във променлива, наречена `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Сега ще дефинираме функцията, която ще извиква Microsoft Learn API за получаване на списък с курсове: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Като добра практика, след това ще проверим дали моделът иска да извика функция. След това ще създадем една от наличните функции и ще я съпоставим с функцията, която се извиква. 
След това ще вземем аргументите на функцията и ще ги съпоставим с аргументите от LLM.

Накрая ще добавим съобщението за извикване на функцията и стойностите, които бяха върнати от съобщението `search_courses`. Това дава на LLM цялата необходима информация, за да
отговори на потребителя с естествен език. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Сега ще изпратим обновеното съобщение до LLM, за да можем да получим отговор на естествен език вместо отговор във формат API JSON. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Кодово предизвикателство 

Отлична работа! За да продължите обучението си по Azure Open AI Function Calling можете да създадете: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Повече параметри на функцията, които могат да помогнат на обучаемите да намерят повече курсове. Достъпните параметри на API можете да намерите тук: 
 - Създайте още един повикване на функция, което приема повече информация от обучаемия, като техния роден език 
 - Създайте обработка на грешки, когато повикването на функцията и/или повикването на API не връща подходящи курсове 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
